# Concepts playground (tiny toy data, runs on the laptop in seconds)
Each section = one idea from the fraud project, done by hand first, then with a library.
Every line has a comment (`#` = comment; Python ignores the rest of the line).

## A. Decision tree from zero (Step 3): Gini, entropy, best split

In [ ]:
import numpy as np                                                # NumPy, nickname np: maths on arrays (np.log2, np.random)
import pandas as pd                                               # pandas, nickname pd: tables

toy_df = pd.DataFrame({                                           # build the 10-row toy table from the lesson
    "amount_high": [1, 1, 1, 0, 1, 0, 0, 0, 0, 0],                # 1 = amount above ₹5,000
    "is_night":    [1, 1, 0, 1, 0, 1, 0, 0, 1, 0],                # 1 = transaction at night
    "is_fraud":    [1, 1, 1, 1, 0, 0, 0, 0, 0, 0],                # the label: rows 1-4 are fraud
})
toy_df                                                            # last line of a cell → Jupyter shows the table

In [ ]:
def gini(labels):                                                 # labels = a column of 0s and 1s
    """Gini impurity = 1 − (p_fraud² + p_legit²). 0 = pure, 0.5 = 50/50 mix."""
    p_fraud = labels.mean()                                       # mean of 0/1 = share of fraud
    p_legit = 1 - p_fraud                                         # the rest is legit
    return 1 - (p_fraud ** 2 + p_legit ** 2)                      # ** = power, so p ** 2 = p squared

def entropy(labels):
    """Entropy = −Σ p·log2(p). 0 = pure, 1 = 50/50 mix (for 2 classes)."""
    p_fraud = labels.mean()
    total = 0.0                                                   # running sum, starts at zero
    for p in [p_fraud, 1 - p_fraud]:                              # loop over the two class shares
        if p > 0:                                                 # log2(0) is undefined; 0·log(0) counts as 0, so skip it
            total -= p * np.log2(p)                               # -= means total = total − (p · log2 p)
    return total

root_labels = toy_df["is_fraud"]                                  # all 10 labels
print(f"root Gini = {gini(root_labels):.3f}, root entropy = {entropy(root_labels):.3f}")  # expect 0.480 and 0.971

In [ ]:
def split_score(df, feature, label="is_fraud", impurity=gini):    # impurity=gini → default; can pass entropy instead
    """Weighted impurity after splitting on a yes/no feature, and the gain."""
    yes_rows = df[df[feature] == 1][label]                        # labels of rows where feature = 1
    no_rows = df[df[feature] == 0][label]                         # labels of rows where feature = 0
    n = len(df)                                                   # total rows
    weighted = (len(yes_rows) / n) * impurity(yes_rows) + (len(no_rows) / n) * impurity(no_rows)  # size-weighted average
    gain = impurity(df[label]) - weighted                         # gain = impurity before − impurity after
    return round(weighted, 3), round(gain, 3)                     # return two numbers (a "tuple")

for feature in ["amount_high", "is_night"]:                      # try both possible questions
    weighted_gini, gini_gain = split_score(toy_df, feature)       # unpack the tuple into two names
    _, info_gain = split_score(toy_df, feature, impurity=entropy)  # _ = "I don't need this value"
    print(f"{feature:12s} weighted Gini={weighted_gini}  Gini gain={gini_gain}  information gain={info_gain}")
# the feature with the bigger gain becomes the first question of the tree → amount_high

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text      # scikit-learn: a ready-made tree + a function that prints it as text

toy_tree = DecisionTreeClassifier(criterion="gini",               # criterion = how to score splits ("gini" or "entropy")
                                  max_depth=2,                    # at most 2 questions from root to leaf
                                  random_state=0)                 # fixed seed for tie-breaking
toy_tree.fit(toy_df[["amount_high", "is_night"]], toy_df["is_fraud"])  # .fit(X, y): learn from features X and labels y
print(export_text(toy_tree, feature_names=["amount_high", "is_night"]))  # "<= 0.50" means "is it 0?" for a yes/no feature
print("P(fraud) per row:", toy_tree.predict_proba(toy_df[["amount_high", "is_night"]])[:, 1].round(2))  # leaf fraud share for each row

In [ ]:
from sklearn.datasets import make_classification                  # makes a synthetic dataset with noise
from sklearn.model_selection import train_test_split              # random split (fine here: no time in toy data)

X_demo, y_demo = make_classification(n_samples=2000,              # 2,000 rows
                                     n_features=10,               # 10 columns
                                     n_informative=4,             # only 4 truly useful columns
                                     flip_y=0.1,                  # 10% labels flipped = noise the tree should NOT memorise
                                     random_state=0)
X_fit, X_check, y_fit, y_check = train_test_split(X_demo, y_demo, test_size=0.3, random_state=0)  # 70% learn, 30% check

for depth in [1, 3, 5, 10, None]:                                 # None = no limit: grow until every leaf is pure
    demo_tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_fit, y_fit)  # make + fit in one line
    train_acc = demo_tree.score(X_fit, y_fit)                     # .score = accuracy (fine here: classes are balanced 50/50)
    check_acc = demo_tree.score(X_check, y_check)
    print(f"max_depth={str(depth):4s} leaves={demo_tree.get_n_leaves():4d} train acc={train_acc:.2f} check acc={check_acc:.2f}")
# deeper tree → train accuracy climbs to 1.00 (memorised), check accuracy stops improving or drops = OVERFITTING

## B. Random forest = bagging + feature randomness (Step 4)
Uses X_fit, y_fit, X_check, y_check made in section A (run section A first).

In [ ]:
rng = np.random.default_rng(0)                                    # NumPy random-number generator with seed 0 (same results every run)
row_ids = np.arange(10)                                           # np.arange(10) = [0 1 2 ... 9] → ids of 10 rows
bootstrap_ids = rng.choice(row_ids, size=10, replace=True)        # pick 10 ids WITH replacement → some repeat, some missing
picked_once_or_more = np.unique(bootstrap_ids)                    # np.unique = the distinct values, sorted
out_of_bag_ids = np.setdiff1d(row_ids, bootstrap_ids)             # np.setdiff1d(a, b) = values in a that are NOT in b → never picked
print("bootstrap sample:", bootstrap_ids)
print("distinct rows used:", len(picked_once_or_more), "| out-of-bag rows:", out_of_bag_ids)

In [ ]:
n_rows = 1000                                                     # a bigger table to check the ~37% rule
oob_shares = []                                                   # [] = empty list; we will append one number per repeat
for repeat in range(200):                                         # range(200) = 0, 1, ..., 199 → do the experiment 200 times
    sample = rng.choice(n_rows, size=n_rows, replace=True)        # one bootstrap sample of row ids
    oob_share = 1 - len(np.unique(sample)) / n_rows               # share of rows never picked
    oob_shares.append(oob_share)                                  # .append = add to the end of the list
print(f"average out-of-bag share = {np.mean(oob_shares):.3f}")    # ≈ 0.368
print(f"theory (1 - 1/n)^n       = {(1 - 1 / n_rows) ** n_rows:.3f}  → 1/e = {1 / np.e:.3f}")

In [ ]:
bagged_probabilities = np.zeros(len(X_check))                     # np.zeros(k) = array of k zeros; we'll add each tree's P(class 1)
n_bagged_trees = 25
for tree_number in range(n_bagged_trees):
    sample = rng.choice(len(X_fit), size=len(X_fit), replace=True)  # bootstrap row positions
    deep_tree = DecisionTreeClassifier(max_depth=None,            # deep tree: low bias, high variance
                                       random_state=tree_number)
    deep_tree.fit(X_fit[sample], y_fit[sample])                   # X_fit[sample] = NumPy "fancy indexing": rows at those positions
    bagged_probabilities += deep_tree.predict_proba(X_check)[:, 1]  # += adds this tree's probabilities to the running total
bagged_probabilities /= n_bagged_trees                            # /= divides by 25 → the AVERAGE probability
bagged_predictions = (bagged_probabilities >= 0.5).astype(int)    # >= 0.5 → True/False → .astype(int) → 1/0

single_tree = DecisionTreeClassifier(max_depth=None, random_state=0).fit(X_fit, y_fit)
print(f"one deep tree   check acc = {single_tree.score(X_check, y_check):.3f}")
print(f"25 bagged trees check acc = {(bagged_predictions == y_check).mean():.3f}")  # mean of True/False = share correct

In [ ]:
from sklearn.ensemble import RandomForestClassifier               # scikit-learn's random forest

for n_trees in [1, 10, 50, 200]:                                  # watch accuracy rise, then plateau (it does NOT overfit)
    forest = RandomForestClassifier(n_estimators=n_trees,         # number of trees
                                    max_features="sqrt",          # each split looks at √(10) ≈ 3 random features
                                    oob_score=(n_trees >= 50),    # OOB score needs enough trees; True only for 50 and 200
                                    n_jobs=-1,                    # -1 = use all CPU cores (trees train in parallel)
                                    random_state=0)
    forest.fit(X_fit, y_fit)
    oob_text = f"OOB acc={forest.oob_score_:.3f}" if n_trees >= 50 else ""  # one-line if/else: value_if_true if condition else value_if_false
    print(f"trees={n_trees:3d}  check acc={forest.score(X_check, y_check):.3f}  {oob_text}")
# OOB accuracy ≈ check accuracy → OOB is a free validation estimate (no separate split needed)

## C. Boosting: AdaBoost idea → gradient boosting (Step 5)
Uses X_fit, y_fit, X_check, y_check from section A.

In [ ]:
error_rate = 0.3                                                  # a stump that is wrong on 30% of (weighted) rows
alpha = 0.5 * np.log((1 - error_rate) / error_rate)               # AdaBoost "say" of this stump; np.log = natural log (ln)
print(f"AdaBoost alpha for err=0.3 → {alpha:.3f}")                # ≈ 0.424; err=0.5 gives 0 (a coin toss gets no say)

In [ ]:
from sklearn.tree import DecisionTreeRegressor                    # a tree that predicts NUMBERS (used to fit residuals)

x_values = np.linspace(0, 6, 80).reshape(-1, 1)                   # 80 points from 0 to 6; reshape(-1, 1) = make it 1 column (sklearn needs 2-D X)
y_values = np.sin(x_values).ravel() + rng.normal(0, 0.1, 80)      # target = sine wave + small noise; .ravel() = flatten to 1-D

learning_rate = 0.1                                               # step size: add only 10% of each tree's correction
current_prediction = np.full(80, y_values.mean())                 # np.full(n, v) = array of n copies of v → F0 = the mean for every point
for round_number in range(1, 101):                                # rounds 1..100 (range stops BEFORE 101)
    residuals = y_values - current_prediction                     # what is still wrong = y − F (= negative gradient of squared loss)
    small_tree = DecisionTreeRegressor(max_depth=2)               # a weak learner: only 2 levels deep
    small_tree.fit(x_values, residuals)                           # the tree learns to predict the RESIDUALS, not y
    current_prediction += learning_rate * small_tree.predict(x_values)  # F_new = F_old + lr × tree's correction
    if round_number in [1, 5, 20, 50, 100]:                       # print only a few rounds
        mse = np.mean((y_values - current_prediction) ** 2)       # mean squared error = average of squared residuals
        print(f"round {round_number:3d}: train MSE = {mse:.4f}")  # keeps falling → the team keeps fixing its own mistakes

In [ ]:
fraud_rate = 0.035                                                # base rate in the fraud data
start_log_odds = np.log(fraud_rate / (1 - fraud_rate))            # log-odds = ln(p / (1 − p)) → the booster's starting value F0
def sigmoid(z):                                                   # converts log-odds back to a probability between 0 and 1
    return 1 / (1 + np.exp(-z))                                   # np.exp = e to the power
print(f"start log-odds = {start_log_odds:.2f} → sigmoid = {sigmoid(start_log_odds):.3f}")
print(f"residual (y − p) for a fraud row = {1 - fraud_rate:.3f}, for a legit row = {0 - fraud_rate:.3f}")  # big push vs tiny push

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier           # scikit-learn's classic gradient boosting

booster = GradientBoostingClassifier(n_estimators=400,            # many trees on purpose, to show overfitting
                                     learning_rate=0.3,           # large steps → overfits sooner
                                     max_depth=3,                 # shallow trees (weak learners)
                                     random_state=0)
booster.fit(X_fit, y_fit)
check_scores = [np.mean(step_pred == y_check)                     # list comprehension: accuracy after each added tree...
                for step_pred in booster.staged_predict(X_check)] # staged_predict = predictions after tree 1, 2, 3, ... 400
for n_trees in [10, 50, 100, 200, 400]:
    print(f"trees={n_trees:3d}  check acc={check_scores[n_trees - 1]:.3f}")  # [n − 1] because Python lists start at index 0
best_n = int(np.argmax(check_scores)) + 1                         # np.argmax = position of the biggest value → best number of trees
print(f"best number of trees = {best_n} → this is what EARLY STOPPING finds automatically (on the valid set)")